# Tema 7 — Gestión de errores
## Laboratorio 5: ejercicio integrado de gestión de errores

**Objetivo**

Combinar validación, `try/except`, `else`, `finally`, `raise`, excepción personalizada y diagnóstico controlado en un flujo pequeño de TI.

**Indicaciones de uso**

Ejecuta las celdas en orden. Algunas celdas incluyen líneas comentadas para provocar errores de forma controlada; descoméntalas solo cuando el instructor lo indique.

> Directorio de trabajo recomendado: `~/Curso_Python/Tema07`  
> Entorno virtual recomendado: `.venv` con prompt `Tema07`.

## 1. Datos de entrada

Se procesan registros de servicios con formato `nombre;puerto;estado`. Algunos registros son incorrectos y deben separarse sin detener todo el proceso.

In [ ]:
lineas = [
    "ssh;22;activo",
    "nginx;443;activo",
    "api;no_numero;activo",
    "backup;70000;activo",
    "rdp;3389;detenido",
    "malformada",
]

for linea in lineas:
    print(linea)

## 2. Excepción personalizada para servicios detenidos

Esta excepción representa un error propio del dominio del ejercicio.

In [ ]:
class ServicioDetenidoError(Exception):
    """Error de dominio: el servicio está registrado, pero no está activo."""
    pass


print("Excepción personalizada definida.")

## 3. Funciones de validación e interpretación

`validar_puerto()` comprueba el rango. `interpretar_linea()` convierte cada línea en un diccionario y lanza `ValueError` si el formato o el puerto no son válidos.

In [ ]:
def validar_puerto(puerto):
    """Valida el rango de un puerto."""
    if not 1 <= puerto <= 65535:
        raise ValueError("Puerto fuera de rango")
    return puerto


def interpretar_linea(linea):
    """
    Convierte una línea de inventario en un diccionario.

    Raises:
        ValueError: si el formato no es correcto o el puerto no es válido.
    """
    partes = linea.split(";")

    if len(partes) != 3:
        raise ValueError("Formato incorrecto")

    nombre, puerto_txt, estado = partes
    nombre = nombre.strip().lower()
    estado = estado.strip().lower()

    puerto = validar_puerto(int(puerto_txt))

    return {
        "nombre": nombre,
        "puerto": puerto,
        "activo": estado == "activo",
    }


print("Funciones de validación definidas.")

## 4. Función de comprobación de disponibilidad

Si el servicio no está activo, se lanza la excepción personalizada `ServicioDetenidoError`.

In [ ]:
def comprobar_disponible(servicio):
    """
    Comprueba si un servicio está activo.

    Raises:
        ServicioDetenidoError: si el servicio está detenido.
    """
    if not servicio["activo"]:
        raise ServicioDetenidoError(f'{servicio["nombre"]} está detenido')
    return servicio


print("Función de disponibilidad definida.")

## 5. Procesar registros con control de errores

El bucle separa servicios válidos, servicios detenidos y registros rechazados. `else` añade los válidos y `finally` simula la limpieza final del recurso.

In [ ]:
servicios_validos = []
rechazados = []
servicios_detenidos = []

for linea in lineas:
    recurso_abierto = False

    try:
        recurso_abierto = True
        servicio = interpretar_linea(linea)
        servicio = comprobar_disponible(servicio)

    except ServicioDetenidoError as error:
        servicios_detenidos.append((linea, str(error)))

    except ValueError as error:
        rechazados.append((linea, str(error)))

    else:
        servicios_validos.append(servicio)

    finally:
        recurso_abierto = False


print("Servicios válidos:", servicios_validos)
print("Servicios detenidos:", servicios_detenidos)
print("Registros rechazados:", rechazados)

## 6. Reporte final

Se imprime el resumen de servicios correctos y los contadores de cada categoría.

In [ ]:
for servicio in servicios_validos:
    print(f'{servicio["nombre"]}:{servicio["puerto"]} -> OK')

print("Total válidos:", len(servicios_validos))
print("Total detenidos:", len(servicios_detenidos))
print("Total rechazados:", len(rechazados))

## Cierre del laboratorio 5

Comprueba que entiendes qué excepción se produce en cada caso, qué bloque la captura y por qué el programa puede continuar ejecutándose.